# CCA-F Review Notebook — Claude Code

This notebook is organized by the provided CCA-F task list. Each task includes a concept explanation, runnable Python example, exam summary, and two practice questions.

## Environment Setup

In [ ]:
import json
import re
from dataclasses import dataclass, field
from datetime import datetime
from typing import Any, Optional

MODEL = "claude-haiku-4-5-20251001"
print("Environment ready: examples are local simulations shaped like Claude API and Claude Code workflows.")

## D3.1 — Configure CLAUDE.md hierarchy and path scoping with YAML frontmatter

### Concept Explanation

`CLAUDE.md` is the primary place for long-lived Claude Code instructions. CCA-F questions usually test three ideas: hierarchy, path scope, and YAML frontmatter. Personal, repository-level, and directory-level rules should be kept distinct; rules closer to the target file are more specific, but temporary task instructions and personal preferences should not be committed as team policy. Path scoping lets different subtrees use different constraints, such as requiring input validation under `src/api/**` while using writing rules under `docs/**`.

YAML frontmatter matters because it makes rule metadata structured: names, included paths, excluded paths, priority, or descriptions can be inspected by tooling. Common exam traps include treating `~/.claude/CLAUDE.md` as team-shared configuration or assuming a directory rule applies to unrelated paths.

In [ ]:
# Task 3.1: CLAUDE.md hierarchy, path scope, and YAML frontmatter.
# This example only simulates configuration parsing; Claude Code is not required.

import fnmatch
from dataclasses import dataclass

claude_md_files = {
    "~/.claude/CLAUDE.md": """---
name: personal-defaults
applies_to: ['**/*']
---
- Use my personal preference for concise answers.
""",
    "./CLAUDE.md": """---
name: project-standards
applies_to: ['**/*']
---
- Respect team architecture boundaries and inspect nearby tests before editing.
""",
    "./src/api/CLAUDE.md": """---
name: api-standards
applies_to: ['src/api/**/*.py', 'src/api/*.py']
exclude: ['src/api/experimental/**']
---
- API handlers must validate input, return structured errors, and cover failure-path tests.
""",
}

def parse_frontmatter(markdown: str) -> tuple[dict, str]:
    """Parse the small frontmatter subset used in this notebook example."""
    if not markdown.startswith("---\n"):
        return {}, markdown
    _, raw_meta, body = markdown.split("---\n", 2)
    meta = {}
    for line in raw_meta.splitlines():
        key, value = line.split(":", 1)
        value = value.strip()
        if value.startswith("["):
            meta[key.strip()] = [item.strip().strip("'") for item in value.strip("[]").split(",") if item.strip()]
        else:
            meta[key.strip()] = value
    return meta, body.strip()

@dataclass(frozen=True)
class ScopedRule:
    source: str
    level: int
    name: str
    applies_to: tuple[str, ...]
    exclude: tuple[str, ...]
    body: str

def load_rules(files: dict[str, str]) -> list[ScopedRule]:
    levels = {"~/.claude/CLAUDE.md": 0, "./CLAUDE.md": 1, "./src/api/CLAUDE.md": 2}
    loaded = []
    for source, markdown in files.items():
        meta, body = parse_frontmatter(markdown)
        loaded.append(ScopedRule(
            source=source,
            level=levels[source],
            name=meta["name"],
            applies_to=tuple(meta.get("applies_to", ["**/*"])),
            exclude=tuple(meta.get("exclude", [])),
            body=body,
        ))
    return sorted(loaded, key=lambda rule: rule.level)

def active_rule_names(path: str) -> list[str]:
    active = []
    for rule in load_rules(claude_md_files):
        included = any(fnmatch.fnmatch(path, pattern) for pattern in rule.applies_to)
        excluded = any(fnmatch.fnmatch(path, pattern) for pattern in rule.exclude)
        if included and not excluded:
            active.append(rule.name)
    return active

for candidate in ["src/api/refund.py", "src/api/experimental/prototype.py", "docs/guide.md"]:
    print(candidate, "=>", active_rule_names(candidate))

### Exam Summary & Common Traps

**Correct patterns**

- Keep team rules in repository-level or directory-level `CLAUDE.md`; keep personal preferences in user-level configuration.
- Use frontmatter or an equivalent structured convention for `applies_to` and `exclude` so path scope is inspectable.
- Store long-lived, cross-task rules in configuration; keep one-off task instructions in the current prompt or issue.

**Common traps**

- Assuming global `~/.claude/CLAUDE.md` is shared with the team.
- Writing vague natural-language policy without saying which paths it applies to.
- Treating directory rules as replacing every higher-level rule; the safer model is stacked rules with local rules adding specificity.

### Practice Questions

**Q1.** A repository needs different Claude Code instructions for `src/api/` and `docs/`. What is the best configuration pattern?

A) Use project/local `CLAUDE.md` files or frontmatter path scope  
B) Put every rule in one user's `~/.claude/CLAUDE.md`  
C) Paste all rules manually before every run  
D) Rely on the model to infer rules from file names

> **Answer: A**  
> **Explanation:** CCA-F favors maintainable, shared project configuration. Path scope lets each directory receive the relevant rules automatically.

**Q2.** Which instruction belongs in a repository-level `CLAUDE.md`?

A) "Change the payment page copy today."  
B) "When editing `src/api/**` handlers, preserve input validation and failure-path tests."  
C) "I personally prefer every answer to be shorter than three sentences."  
D) "Ignore lint and test failures in this repository."

> **Answer: B**  
> **Explanation:** Repository rules should be durable, shared, and verifiable. Temporary tasks and personal style preferences do not belong there.

## D3.2 — Create custom slash commands and manage .claude/commands/

### Concept Explanation

Slash commands package repeatable team prompts as commands, usually under the repository's `.claude/commands/` directory, such as `.claude/commands/review-api.md`. Their value is consistent workflow invocation: argument expectations, output format, review checklist, and completion criteria. Team-shared commands should be version controlled; user-local commands are for personal shortcuts.

`SKILL.md` is better for a fuller reusable capability and commonly includes YAML frontmatter such as `name`, `description`, `argument-hint`, `allowed-tools`, and context isolation. CCA-F tests the boundary: a slash command is an invocation shortcut, while a Skill packages capability. Skills should use least privilege and avoid exposing unrelated context or broad tools by default.

In [ ]:
# Task 3.2: slash commands, .claude/commands, and Skill isolation.
# This example simulates command discovery and Skill frontmatter checks without Claude Code.

project_files = {
    ".claude/commands/review-api.md": """# /review-api
Argument: $ARGUMENTS

Review the API file for auth, validation, structured errors, and tests.
Return findings as severity, file, line, and fix.
""",
    "skills/api-review/SKILL.md": """---
name: api-review
description: Review API handlers with isolated context.
argument-hint: path to an API module
allowed-tools: [Read, Grep]
context: fork
---

Inspect only the requested API module and adjacent tests.
""",
}

def discover_slash_commands(files: dict[str, str]) -> dict[str, str]:
    commands = {}
    for path, body in files.items():
        if path.startswith(".claude/commands/") and path.endswith(".md"):
            command_name = "/" + path.rsplit("/", 1)[-1].removesuffix(".md")
            commands[command_name] = body
    return commands

def parse_inline_frontmatter(markdown: str) -> dict[str, object]:
    meta_text = markdown.split("---", 2)[1]
    meta = {}
    for line in meta_text.splitlines():
        if not line.strip():
            continue
        key, value = line.split(":", 1)
        value = value.strip()
        if value.startswith("["):
            meta[key] = [item.strip() for item in value.strip("[]").split(",")]
        else:
            meta[key] = value
    return meta

def skill_is_exam_ready(skill_markdown: str) -> bool:
    meta = parse_inline_frontmatter(skill_markdown)
    required = {"name", "description", "argument-hint", "allowed-tools", "context"}
    least_privilege = set(meta.get("allowed-tools", [])) <= {"Read", "Grep", "Glob"}
    isolated = meta.get("context") in {"fork", "isolated"}
    return required <= set(meta) and least_privilege and isolated

commands = discover_slash_commands(project_files)
print("Available commands:", sorted(commands))
print("/review-api has argument placeholder:", "$ARGUMENTS" in commands["/review-api"])
print("Skill is least-privilege and isolated:", skill_is_exam_ready(project_files["skills/api-review/SKILL.md"]))

### Exam Summary & Common Traps

**Correct patterns**

- Put team-shared slash commands in repository `.claude/commands/`, with arguments, output format, and completion criteria.
- Put larger reusable capabilities in `SKILL.md`, using frontmatter for capability description, argument hints, allowed tools, and context isolation.
- Follow least privilege for Skill tools; if `Read/Grep/Glob` is enough, do not grant shell or network access by default.

**Common traps**

- Putting shared commands in personal `~/.claude/commands/`, making them unreproducible for teammates or CI.
- Treating slash commands and Skills as the same abstraction: commands are shortcuts, Skills are capability packages.
- Omitting argument hints, output expectations, or isolation, which increases context pollution and permission risk.

### Practice Questions

**Q1.** A team wants every member to use the same `/review-api` command. Where should it live?

A) Repository `.claude/commands/review-api.md`  
B) Each user's `~/.claude/commands/review-api.md`  
C) A section in the root `README.md` only  
D) The CI platform's secrets page

> **Answer: A**  
> **Explanation:** Repository `.claude/commands/` files can be version controlled, shared, and reviewed.

**Q2.** Which `SKILL.md` frontmatter design best matches CCA-F expectations?

A) Only define `name`; explain everything else in prose later.  
B) Declare `description`, `argument-hint`, least-privilege `allowed-tools`, and isolated context.  
C) Grant Bash, network access, and the entire repository by default.  
D) Omit frontmatter and let the model infer tools and boundaries.

> **Answer: B**  
> **Explanation:** Skills should be discoverable, constrained, and isolated. Broad tools and implicit boundaries are common distractors.

## D3.3 — Use plan mode and session management commands

### Concept Explanation

Plan Mode is for exploring, proposing an approach, and waiting for confirmation before execution. It fits multi-file, high-risk, or architecture-boundary work. It is not mandatory for every task; a single-file, low-risk, well-scoped fix can go straight to execution. CCA-F scenario questions often test whether you choose workflow based on risk, blast radius, and recoverability.

Session management covers distinct mechanisms: `/memory` stores durable preferences or team conventions, not temporary task details; `/compact` compresses long conversations while preserving key decisions; session continuity is about resuming useful task context across sessions; `--print` is for non-interactive script and CI usage. These commands are not interchangeable.

In [ ]:
# Task 3.3: Plan Mode, /memory, /compact, continuity, and --print.
# This example recommends workflow from task risk without calling Claude Code.

from dataclasses import dataclass, field

@dataclass
class ClaudeCodeSession:
    memory: list[str] = field(default_factory=list)
    transcript_tokens: int = 0
    decisions: list[str] = field(default_factory=list)

    def remember(self, fact: str, durable: bool) -> None:
        # /memory should store durable facts, not one-off issue details.
        if durable:
            self.memory.append(fact)

    def maybe_compact(self) -> str:
        if self.transcript_tokens > 150_000:
            kept = "; ".join(self.decisions[-3:]) or "preserve current goal and next step"
            self.transcript_tokens = 40_000
            return f"/compact -> {kept}"
        return "No /compact needed"

def choose_execution_mode(files_touched: int, risk: str, in_ci: bool) -> str:
    if in_ci:
        return 'claude --print "run review and return JSON" --output-format json'
    if files_touched > 10 or risk == "high":
        return "Plan Mode: propose approach, risks, and verification before editing"
    return "Direct execution: scope is clear; edit and verify"

session = ClaudeCodeSession(transcript_tokens=180_000, decisions=["preserve public API", "add tests before implementation"])
session.remember("API handlers in this repo must cover failure-path tests", durable=True)
session.remember("The current PR number is 1287", durable=False)

print("Interactive large change:", choose_execution_mode(files_touched=18, risk="high", in_ci=False))
print("CI invocation:", choose_execution_mode(files_touched=3, risk="medium", in_ci=True))
print("Compaction result:", session.maybe_compact())
print("memory:", session.memory)

### Exam Summary & Common Traps

**Correct patterns**

- Use Plan Mode for multi-file, high-risk, architecture-level changes before editing.
- Use `/compact` for long-context compression; use `/memory` only for durable facts; use session continuity to resume task state.
- Use `--print` or `-p` for non-interactive CI and scripts.

**Common traps**

- Forcing a long plan for every tiny fix.
- Storing temporary issue details in `/memory`, polluting later tasks.
- Using `/compact` as a planning tool, or using Plan Mode to solve CI non-interactivity.
- Dropping long context outright instead of compacting key decisions.

### Practice Questions

**Q1.** A task refactors authentication boundaries, touches more than twenty files, and may alter a public API. What should happen first?

A) Use Plan Mode to explore dependencies, risks, and verification before editing  
B) Batch-edit every file and inspect tests afterward  
C) Run `/compact`, then skip approach confirmation  
D) Only run the formatter

> **Answer: A**  
> **Explanation:** High-risk, multi-file, architecture-level work should be planned and confirmed before execution.

**Q2.** Which item belongs in `/memory`?

A) Temporary reproduction steps for the current issue  
B) "All API handlers in this repository must include failure-path tests."  
C) The full terminal log from the last command  
D) What the next assistant message should say

> **Answer: B**  
> **Explanation:** `/memory` is for durable, cross-session team conventions. Temporary context belongs in the active session or task record.

## D3.4 — Integrate Claude Code into CI/CD pipelines

### Concept Explanation

Claude Code in CI/CD must be headless, non-interactive, parseable, and able to fail. `--print` or `-p` makes the command emit output and exit; structured output such as JSON lets the pipeline read conclusions, failure reasons, and repair suggestions. Producing natural-language advice alone is not a CI integration because the pipeline cannot reliably decide pass or fail.

A reliable CI/CD design also needs triggers and a verification loop: for example, run on pull requests, feed test failures into the next repair attempt, then check exit codes and test status. Common exam distractors include raising temperature, adding tokens, or putting an interactive Claude Code session directly into CI.

In [ ]:
# Task 3.4: Claude Code CI/CD, headless mode, --print, and verification loops.
# This example simulates CI commands and test feedback without requiring Claude Code.

import json
import shlex

ci_workflow = {
    "on": ["pull_request"],
    "steps": [
        {
            "name": "claude-review",
            "run": 'claude --print "Review changed files and return JSON" --output-format json',
        },
        {
            "name": "tests",
            "run": "pytest -q",
        },
    ],
}

def is_non_interactive_claude_step(command: str) -> bool:
    tokens = shlex.split(command)
    return "claude" in tokens and ("--print" in tokens or "-p" in tokens) and "--output-format" in tokens and "json" in tokens

def simulate_test_verification_loop(test_results: list[bool]) -> dict[str, object]:
    attempts = []
    for attempt, passed in enumerate(test_results, start=1):
        attempts.append({"attempt": attempt, "tests_passed": passed})
        if passed:
            return {"status": "pass", "attempts": attempts, "exit_code": 0}
    return {"status": "fail", "attempts": attempts, "exit_code": 1}

review_step = ci_workflow["steps"][0]["run"]
result = simulate_test_verification_loop([False, False, True])

print("Claude step is CI-safe:", is_non_interactive_claude_step(review_step))
print(json.dumps(result, indent=2))
assert is_non_interactive_claude_step(review_step)
assert result["exit_code"] == 0

### Exam Summary & Common Traps

**Correct patterns**

- Use `claude --print` or `claude -p` in CI so the command exits after producing output.
- Request structured output such as JSON, and have the pipeline check exit codes, fields, and test results.
- Trigger work from explicit events such as pull requests, pushes, or scheduled jobs.
- Feed test failure summaries into repair loops and let verification decide pass or fail.

**Common traps**

- Starting an interactive session in CI, causing the job to hang.
- Emitting prose only, leaving the pipeline with nothing stable to parse.
- Ignoring exit codes or test failures and treating "the model said it fixed it" as success.
- Trying to replace headless execution and verification with higher temperature or more tokens.

### Practice Questions

**Q1.** A Claude Code CI job hangs while waiting for input. What is the direct fix?

A) Use `claude --print` or `claude -p` for non-interactive execution  
B) Raise temperature so the model is more proactive  
C) Increase max tokens so the model has more space  
D) Point stdin at a developer keyboard

> **Answer: A**  
> **Explanation:** CI/CD must be headless. `--print` / `-p` is the key non-interactive execution mode.

**Q2.** Which CI design best matches CCA-F expectations?

A) Trigger Claude Code review on PRs, emit JSON, run tests, feed failure summaries into repair, then check exit status  
B) Open Claude Code manually, paste logs, and verbally notify the team  
C) Ask the model for a prose-only "looks good" summary  
D) Merge despite failing tests because the model attempted a repair

> **Answer: A**  
> **Explanation:** Production CI integration must be non-interactive, structured, verifiable, and able to fail.